In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import sys
sys.path.append('..')
from environments.line_world import LineWorldEnv

from algorithms.dynamic_programming import (
    iterative_policy_evaluation,
    policy_iteration,
    value_iteration
)

In [2]:
S = LineWorldEnv.S
A = LineWorldEnv.A
R = LineWorldEnv.R
T = LineWorldEnv.T
p = LineWorldEnv.p

print("MDP LineWorld défini ✅")
print(f"États : {S}")
print(f"Actions : {A} (0=Gauche, 1=Droite)")
print(f"Rewards : {R}")
print(f"États terminaux : {T}")

NameError: name 'LineWorldEnv' is not defined

In [ ]:
pi_right = np.zeros((len(S), len(A)))
pi_right[:, 1] = 1.0

pi_left = np.zeros((len(S), len(A)))
pi_left[:, 0] = 1.0

pi_random = np.ones((len(S), len(A))) / len(A)

pi_mostly_right = np.zeros((len(S), len(A)))
pi_mostly_right[:, 0] = 0.2
pi_mostly_right[:, 1] = 0.8

V_right = iterative_policy_evaluation(S, A, R, p, T, pi_right)
V_left = iterative_policy_evaluation(S, A, R, p, T, pi_left)
V_random = iterative_policy_evaluation(S, A, R, p, T, pi_random)
V_mostly_right = iterative_policy_evaluation(S, A, R, p, T, pi_mostly_right)

print(f"V toujours Droite     : {V_right.round(3)}")
print(f"V toujours Gauche     : {V_left.round(3)}")
print(f"V aléatoire           : {V_random.round(3)}")
print(f"V mostly Droite (80%) : {V_mostly_right.round(3)}")

In [ ]:
states = ['État 0', 'État 1', 'État 2', 'État 3', 'État 4']
x = np.arange(len(states))
width = 0.2

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - 1.5*width, V_right,        width, label='Toujours Droite', color='green')
ax.bar(x - 0.5*width, V_mostly_right, width, label='Mostly Droite 80%', color='blue')
ax.bar(x + 0.5*width, V_random,       width, label='Aléatoire', color='orange')
ax.bar(x + 1.5*width, V_left,         width, label='Toujours Gauche', color='red')

ax.set_xlabel('États')
ax.set_ylabel('Valeur V(s)')
ax.set_title('Comparaison des policies — iterative_policy_evaluation — LineWorld')
ax.set_xticks(x)
ax.set_xticklabels(states)
ax.legend()
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
t1 = time.time()
pi_pi, V_pi = policy_iteration(S, A, R, p, T)
t1 = time.time() - t1

t2 = time.time()
pi_vi, V_vi = value_iteration(S, A, R, p, T)
t2 = time.time() - t2

print("=== Policy Iteration vs Value Iteration ===")
print(f"Policy Iteration : V = {V_pi.round(3)} | temps = {t1:.4f}s")
print(f"Value Iteration  : V = {V_vi.round(3)} | temps = {t2:.4f}s")
print(f"\nPolicy PI : {['Droite' if a==1 else 'Gauche' for a in pi_pi.argmax(axis=1)]}")
print(f"Policy VI : {['Droite' if a==1 else 'Gauche' for a in pi_vi.argmax(axis=1)]}")

In [ ]:
gammas = [0.1, 0.5, 0.9, 0.99, 0.999]

fig, ax = plt.subplots(figsize=(10, 5))
for g in gammas:
    _, V = value_iteration(S, A, R, p, T, gamma=g)
    ax.plot(S, V, marker='o', label=f'gamma={g}')

ax.set_xlabel('États')
ax.set_ylabel('Valeur V(s)')
ax.set_title('Impact de gamma — LineWorld')
ax.legend()
ax.set_xticks(S)
plt.tight_layout()
plt.show()

In [ ]:
thetas = [0.1, 0.01, 0.001, 0.0001, 0.00001]
times = []
V_results = []

for theta in thetas:
    t = time.time()
    _, V = value_iteration(S, A, R, p, T, theta=theta)
    times.append(time.time() - t)
    V_results.append(V.copy())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(thetas, times, marker='o', color='blue')
ax1.set_xlabel('Theta')
ax1.set_ylabel('Temps (s)')
ax1.set_title('Impact de theta sur le temps')
ax1.set_xscale('log')

for i, theta in enumerate(thetas):
    ax2.plot(S, V_results[i], marker='o', label=f'theta={theta}')
ax2.set_xlabel('États')
ax2.set_ylabel('Valeur V(s)')
ax2.set_title('Impact de theta sur la précision')
ax2.legend()

plt.tight_layout()
plt.show()